# Carga controlada del manifest Drive -> GCS

## Objetivo y riesgos

Este notebook implementa una carga controlada, reanudable y protegida contra sobrescrituras desde Google Drive hacia Google Cloud Storage, usando el manifest validado por los notebooks 38 y 39.

Garantias de esta etapa:

- Por defecto no ejecuta escrituras remotas.
- No sobrescribe objetos existentes.
- No elimina objetos remotos.
- Cada carga exige una precondicion de generacion en GCS.
- La primera etapa incluye unicamente metadata E5 y metadata E9.
- SPIDER y AXIAL quedan fuera de esta primera ejecucion.

## Configuracion segura

La carga solo queda armada cuando `UPLOAD_ENABLED` es exactamente `True` y `CONFIRM_UPLOAD` coincide con el token dinamico impreso por el notebook. Con los valores predeterminados, el notebook valida y prepara el lote, pero no escribe en GCS.

In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import tempfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd

UPLOAD_ENABLED = True
CONFIRM_UPLOAD = "UPLOAD_2_FILES_422816_BYTES_TO_pfi-rm-lumbar-artifacts-2026-ef_fa54c89a278d"
TARGET_COMPONENTS = ("metadata_e5", "metadata_e9")
MAX_FILES_THIS_RUN = 2
FAIL_FAST = True

PROJECT_ID = "pfi-asplanatti-fabrello-v1"
BUCKET_NAME = "pfi-rm-lumbar-artifacts-2026-ef"
BUCKET_URI_PREFIX = f"gs://{BUCKET_NAME}/"
GCS_PREFIX = "datasets/"
ALLOWED_ZERO_BYTE_PLACEHOLDERS = {GCS_PREFIX}
EXPECTED_BUCKET_LOCATION = "US-CENTRAL1"
EXPECTED_BUCKET_STORAGE_CLASS = "STANDARD"

DRIVE_ROOT = Path("/content/drive")
MY_DRIVE = DRIVE_ROOT / "MyDrive"
PFI_ROOT = MY_DRIVE / "PFI_MVP"
PLAN_DIR = PFI_ROOT / "results" / "GCS_dataset_migration_plan"
DRY_RUN_DIR = PFI_ROOT / "results" / "GCS_dataset_upload_dry_run"
UPLOAD_STATE_DIR = PFI_ROOT / "results" / "GCS_dataset_upload"

MANIFEST_CSV = PLAN_DIR / "gcs_upload_manifest.csv"
DRY_RUN_SUMMARY_JSON = DRY_RUN_DIR / "gcs_dry_run_summary.json"
DRY_RUN_OBJECTS_CSV = DRY_RUN_DIR / "gcs_dry_run_objects.csv"
RECEIPT_CSV = UPLOAD_STATE_DIR / "gcs_upload_receipt.csv"
STATE_JSON = UPLOAD_STATE_DIR / "gcs_upload_state.json"
FAILURES_CSV = UPLOAD_STATE_DIR / "gcs_upload_failures.csv"

EXPECTED_MANIFEST_ROWS = 2058
EXPECTED_TOTAL_BYTES = 3922288649
EXPECTED_INITIAL_SELECTED_BYTES = 422816
EXPECTED_MANIFEST_COLUMNS = [
    "component",
    "source_path",
    "source_root",
    "relative_path",
    "destination_uri",
    "suffix",
    "size_bytes",
    "modified_at",
    "sha256",
    "referenced_rows",
    "exists",
    "is_symlink",
    "status",
]
EXPECTED_DRY_RUN_SUMMARY = {
    "DRY_RUN": True,
    "manifest_rows": EXPECTED_MANIFEST_ROWS,
    "manifest_bytes": EXPECTED_TOTAL_BYTES,
    "to_upload": EXPECTED_MANIFEST_ROWS,
    "exists_same_size_unverified": 0,
    "conflict_size": 0,
    "unplanned_existing": 0,
    "allowed_zero_byte_placeholders": 1,
    "GCS_DRY_RUN_READY": True,
}
RECEIPT_COLUMNS = [
    "manifest_sha256",
    "component",
    "source_path",
    "relative_path",
    "destination_uri",
    "object_name",
    "size_bytes",
    "crc32c",
    "md5_hash",
    "generation",
    "updated",
    "uploaded_at_utc",
    "upload_status",
]
FAILURE_COLUMNS = [
    "manifest_sha256",
    "component",
    "relative_path",
    "destination_uri",
    "error_type",
    "error_message",
    "failed_at_utc",
]

print(json.dumps({
    "UPLOAD_ENABLED": UPLOAD_ENABLED,
    "CONFIRM_UPLOAD_SET": bool(CONFIRM_UPLOAD),
    "TARGET_COMPONENTS": TARGET_COMPONENTS,
    "MAX_FILES_THIS_RUN": MAX_FILES_THIS_RUN,
    "bucket": BUCKET_NAME,
    "state_dir": str(UPLOAD_STATE_DIR),
}, indent=2))

{
  "UPLOAD_ENABLED": true,
  "CONFIRM_UPLOAD_SET": true,
  "TARGET_COMPONENTS": [
    "metadata_e5",
    "metadata_e9"
  ],
  "MAX_FILES_THIS_RUN": 2,
  "bucket": "pfi-rm-lumbar-artifacts-2026-ef",
  "state_dir": "/content/drive/MyDrive/PFI_MVP/results/GCS_dataset_upload"
}


## Montaje de Drive

Reutiliza `/content/drive/MyDrive` si ya existe. No fuerza remontaje ni limpia directorios.

In [2]:
def ensure_drive_available() -> None:
    if MY_DRIVE.exists():
        print(f"Drive ya disponible: {MY_DRIVE}")
    else:
        try:
            from google.colab import drive
        except ImportError as exc:
            raise RuntimeError("Este notebook debe ejecutarse en Colab o con /content/drive/MyDrive disponible.") from exc
        drive.mount(str(DRIVE_ROOT))
    if not PFI_ROOT.exists():
        raise FileNotFoundError(f"No existe PFI_ROOT: {PFI_ROOT}")
    print(f"PFI_ROOT OK: {PFI_ROOT}")

ensure_drive_available()

Drive ya disponible: /content/drive/MyDrive
PFI_ROOT OK: /content/drive/MyDrive/PFI_MVP


## Validacion del manifest y dry-run

Vuelve a validar el manifest completo y exige que la evidencia read-only del notebook 39 coincida con los valores congelados antes de continuar.

In [3]:
def require_readable_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Falta archivo requerido: {path}")
    if not path.is_file():
        raise ValueError(f"No es archivo regular: {path}")
    with path.open("rb") as fh:
        fh.read(1)


def sha256_stream(path: Path) -> str:
    with path.open("rb") as fh:
        return hashlib.file_digest(fh, "sha256").hexdigest()


def parse_bool_series(series: pd.Series, name: str) -> pd.Series:
    if series.dtype == bool:
        return series
    normalized = series.astype(str).str.strip().str.lower()
    bad = sorted(set(normalized) - {"true", "false"})
    if bad:
        raise ValueError(f"Columna {name} contiene booleanos invalidos: {bad[:10]}")
    return normalized == "true"


def path_is_inside(path_text: str, root: Path) -> bool:
    path = Path(path_text)
    return path.is_absolute() and (path == root or root in path.parents)


def destination_object_name(uri: str) -> str:
    if not isinstance(uri, str) or not uri.startswith(BUCKET_URI_PREFIX):
        raise ValueError(f"Destino fuera de bucket: {uri}")
    object_name = uri[len(BUCKET_URI_PREFIX):]
    if not object_name.startswith(GCS_PREFIX):
        raise ValueError(f"Destino fuera de prefijo {GCS_PREFIX}: {uri}")
    return object_name


def validate_manifest() -> tuple[pd.DataFrame, str]:
    require_readable_file(MANIFEST_CSV)
    manifest_sha256 = sha256_stream(MANIFEST_CSV)
    manifest = pd.read_csv(MANIFEST_CSV, keep_default_na=False)
    if list(manifest.columns) != EXPECTED_MANIFEST_COLUMNS:
        raise ValueError(f"Columnas inesperadas: {list(manifest.columns)}")
    if len(manifest) != EXPECTED_MANIFEST_ROWS:
        raise ValueError(f"Filas inesperadas: {len(manifest)} != {EXPECTED_MANIFEST_ROWS}")
    manifest["size_bytes"] = pd.to_numeric(manifest["size_bytes"], errors="raise").astype("int64")
    manifest["exists"] = parse_bool_series(manifest["exists"], "exists")
    manifest["is_symlink"] = parse_bool_series(manifest["is_symlink"], "is_symlink")
    if int(manifest["size_bytes"].sum()) != EXPECTED_TOTAL_BYTES:
        raise ValueError("Total de bytes del manifest no coincide con el valor congelado")
    if not bool((manifest["status"].astype(str) == "OK").all()):
        raise ValueError("Hay filas con status distinto de OK")
    if not bool(manifest["exists"].all()):
        raise ValueError("Hay filas con exists != True")
    if bool(manifest["is_symlink"].any()):
        raise ValueError("Hay filas marcadas como symlink")
    if bool((manifest["size_bytes"] <= 0).any()):
        raise ValueError("Hay filas con size_bytes <= 0")
    if manifest["destination_uri"].duplicated().any():
        raise ValueError("Hay destination_uri duplicados")
    if manifest.duplicated(["component", "source_path"]).any():
        raise ValueError("Hay source_path duplicado dentro de componente")
    if bool(manifest["relative_path"].astype(str).str.strip().eq("").any()):
        raise ValueError("Hay relative_path vacios")
    if bool(manifest["relative_path"].astype(str).str.split("/").map(lambda parts: ".." in parts).any()):
        raise ValueError('Hay relative_path con segmento ".."')
    if bool(manifest["suffix"].astype(str).str.lower().eq(".zip").any()):
        raise ValueError("Hay ZIP incluidos")
    if not bool(manifest["source_path"].map(lambda value: path_is_inside(str(value), PFI_ROOT)).all()):
        raise ValueError("Hay source_path fuera de PFI_ROOT")
    if not bool(manifest["destination_uri"].astype(str).str.startswith(BUCKET_URI_PREFIX).all()):
        raise ValueError("Hay destinos fuera del bucket esperado")
    print(json.dumps({"manifest_rows": len(manifest), "manifest_sha256": manifest_sha256}, indent=2))
    return manifest, manifest_sha256


def validate_dry_run_evidence() -> dict[str, Any]:
    require_readable_file(DRY_RUN_SUMMARY_JSON)
    require_readable_file(DRY_RUN_OBJECTS_CSV)
    with DRY_RUN_SUMMARY_JSON.open("r", encoding="utf-8") as fh:
        summary = json.load(fh)
    for key, expected in EXPECTED_DRY_RUN_SUMMARY.items():
        actual = summary.get(key)
        if actual != expected:
            raise ValueError(f"Dry-run summary invalido para {key}: {actual!r} != {expected!r}")
    print(json.dumps({"dry_run_summary": "OK", "GCS_DRY_RUN_READY": summary.get("GCS_DRY_RUN_READY")}, indent=2))
    return summary

manifest_df, MANIFEST_SHA256 = validate_manifest()
dry_run_summary = validate_dry_run_evidence()
LOCAL_VALIDATIONS_READY = True

{
  "manifest_rows": 2058,
  "manifest_sha256": "fa54c89a278d850021c0f91c0a27b3b5211c86301c9e4f125d75d517f39b793b"
}
{
  "dry_run_summary": "OK",
  "GCS_DRY_RUN_READY": true
}


## Revalidacion inmediata de fuentes

Antes de seleccionar el lote, confirma existencia, archivo regular, ausencia de symlink y tama?o exacto. Para metadata E5/E9 recalcula SHA-256 y lo compara contra el manifest.

In [4]:
def revalidate_sources_now(manifest: pd.DataFrame) -> None:
    total = len(manifest)
    for index, row in enumerate(manifest.itertuples(index=False), start=1):
        source_path = Path(row.source_path)
        if not source_path.exists():
            raise FileNotFoundError(f"Fuente faltante: component={row.component} relative_path={row.relative_path}")
        if source_path.is_symlink():
            raise ValueError(f"Symlink rechazado: component={row.component} relative_path={row.relative_path}")
        if not source_path.is_file():
            raise ValueError(f"Fuente no regular: component={row.component} relative_path={row.relative_path}")
        current_size = source_path.stat().st_size
        if current_size != int(row.size_bytes):
            raise ValueError(f"Size mismatch: component={row.component} relative_path={row.relative_path}")
        if row.component in TARGET_COMPONENTS:
            actual_sha = sha256_stream(source_path)
            expected_sha = str(row.sha256).strip()
            if actual_sha != expected_sha:
                raise ValueError(f"SHA mismatch: component={row.component} relative_path={row.relative_path}")
        if index % 250 == 0 or index == total:
            print(f"Fuentes revalidadas: {index}/{total}")

revalidate_sources_now(manifest_df)
SOURCE_FILES_READY = True

Fuentes revalidadas: 250/2058
Fuentes revalidadas: 500/2058
Fuentes revalidadas: 750/2058
Fuentes revalidadas: 1000/2058
Fuentes revalidadas: 1250/2058
Fuentes revalidadas: 1500/2058
Fuentes revalidadas: 1750/2058
Fuentes revalidadas: 2000/2058
Fuentes revalidadas: 2058/2058


## Autenticacion GCP

Usa autenticacion interactiva de Colab y librerias oficiales. No usa archivos JSON de credenciales ni imprime tokens.

In [5]:
def authenticate_gcp():
    try:
        from google.colab import auth
    except ImportError as exc:
        raise RuntimeError("La autenticacion interactiva requiere Google Colab.") from exc
    try:
        import google.auth
        from google.cloud import storage
    except ImportError as exc:
        raise RuntimeError("Falta google-cloud-storage/google-auth. Instalar manualmente antes de ejecutar esta celda.") from exc
    auth.authenticate_user()
    credentials, active_project = google.auth.default()
    client = storage.Client(project=PROJECT_ID, credentials=credentials)
    print(json.dumps({"client_project": PROJECT_ID, "auth_project_detected": active_project}, indent=2))
    return client

storage_client = authenticate_gcp()

{
  "client_project": "pfi-asplanatti-fabrello-v1",
  "auth_project_detected": ""
}


## Inspeccion del bucket

Valida metadata del bucket, lista una sola vez `datasets/` y permite solamente el marcador exacto `datasets/` si su tama?o es cero.

In [6]:
def inspect_bucket(client) -> tuple[Any, dict[str, dict[str, Any]], list[dict[str, Any]], list[dict[str, Any]]]:
    bucket = client.bucket(BUCKET_NAME)
    bucket.reload(client=client)
    location = str(bucket.location or "").upper()
    storage_class = str(bucket.storage_class or "").upper()
    if bucket.name != BUCKET_NAME:
        raise ValueError(f"Bucket inesperado: {bucket.name}")
    if location != EXPECTED_BUCKET_LOCATION:
        raise ValueError(f"Ubicacion inesperada: {location}")
    if storage_class != EXPECTED_BUCKET_STORAGE_CLASS:
        raise ValueError(f"Storage class inesperada: {storage_class}")

    existing: dict[str, dict[str, Any]] = {}
    placeholders: list[dict[str, Any]] = []
    external_objects: list[dict[str, Any]] = []
    for blob in client.list_blobs(BUCKET_NAME, prefix=GCS_PREFIX):
        item = {
            "name": blob.name,
            "size": int(blob.size or 0),
            "crc32c": blob.crc32c,
            "md5_hash": blob.md5_hash,
            "generation": str(blob.generation) if blob.generation is not None else None,
            "updated": blob.updated.isoformat() if blob.updated is not None else None,
        }
        existing[blob.name] = item
        if blob.name in ALLOWED_ZERO_BYTE_PLACEHOLDERS and item["size"] == 0:
            placeholders.append(item)
        elif blob.name not in set(manifest_df["destination_uri"].map(destination_object_name)):
            external_objects.append(item)
    if external_objects:
        raise RuntimeError(f"Objetos no planificados bajo {GCS_PREFIX}: {external_objects[:10]}")
    print(json.dumps({
        "bucket": bucket.name,
        "location": location,
        "storage_class": storage_class,
        "objects_under_prefix": len(existing),
        "placeholder_count": len(placeholders),
    }, indent=2))
    return bucket, existing, placeholders, external_objects

bucket, existing_objects, allowed_placeholders, unplanned_existing_objects = inspect_bucket(storage_client)
BUCKET_READY = True

{
  "bucket": "pfi-rm-lumbar-artifacts-2026-ef",
  "location": "US-CENTRAL1",
  "storage_class": "STANDARD",
  "objects_under_prefix": 3,
  "placeholder_count": 1
}


## Estado reanudable

Carga recibos y fallas previas desde Drive. El recibo queda ligado al SHA-256 del manifest para evitar mezclar ejecuciones incompatibles.

In [7]:
def empty_receipt_df() -> pd.DataFrame:
    return pd.DataFrame(columns=RECEIPT_COLUMNS)


def empty_failures_df() -> pd.DataFrame:
    return pd.DataFrame(columns=FAILURE_COLUMNS)


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp_name = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=str(path.parent))
    try:
        with os.fdopen(fd, "w", encoding="utf-8", newline="") as fh:
            fh.write(text)
            fh.flush()
            os.fsync(fh.fileno())
        os.replace(tmp_name, path)
    finally:
        tmp_path = Path(tmp_name)
        if tmp_path.exists():
            tmp_path.unlink()


def save_receipt(receipt: pd.DataFrame) -> None:
    atomic_write_text(RECEIPT_CSV, receipt[RECEIPT_COLUMNS].to_csv(index=False, quoting=csv.QUOTE_MINIMAL))


def save_failures(failures: pd.DataFrame) -> None:
    atomic_write_text(FAILURES_CSV, failures[FAILURE_COLUMNS].to_csv(index=False, quoting=csv.QUOTE_MINIMAL))


def save_state(payload: dict[str, Any]) -> None:
    atomic_write_text(STATE_JSON, json.dumps(payload, indent=2, ensure_ascii=False) + "\n")


def load_receipt() -> pd.DataFrame:
    if not RECEIPT_CSV.exists():
        return empty_receipt_df()
    receipt = pd.read_csv(RECEIPT_CSV, keep_default_na=False)
    if list(receipt.columns) != RECEIPT_COLUMNS:
        raise ValueError(f"Columnas inesperadas en receipt: {list(receipt.columns)}")
    if not receipt.empty and not bool((receipt["manifest_sha256"] == MANIFEST_SHA256).all()):
        raise ValueError("Receipt ligado a otro manifest_sha256")
    if receipt["destination_uri"].duplicated().any():
        raise ValueError("Receipt contiene destination_uri duplicados")
    return receipt


def load_failures() -> pd.DataFrame:
    if not FAILURES_CSV.exists():
        return empty_failures_df()
    failures = pd.read_csv(FAILURES_CSV, keep_default_na=False)
    if list(failures.columns) != FAILURE_COLUMNS:
        raise ValueError(f"Columnas inesperadas en failures: {list(failures.columns)}")
    return failures

receipt_df = load_receipt()
failures_df = load_failures()
print(json.dumps({"receipt_rows": len(receipt_df), "failure_rows": len(failures_df)}, indent=2))

{
  "receipt_rows": 2,
  "failure_rows": 0
}


## Seleccion del lote

Clasifica objetos ya verificados por recibo, bloquea objetos existentes no verificables y selecciona solo metadata E5/E9, respetando el orden del manifest y el limite del lote.

In [8]:
def receipt_lookup(receipt: pd.DataFrame) -> dict[str, dict[str, Any]]:
    return {str(row.destination_uri): row._asdict() for row in receipt.itertuples(index=False)}


def classify_manifest_rows(manifest: pd.DataFrame, existing: dict[str, dict[str, Any]], receipt: pd.DataFrame) -> pd.DataFrame:
    receipts = receipt_lookup(receipt)
    rows: list[dict[str, Any]] = []
    for ordinal, row in enumerate(manifest.itertuples(index=False)):
        object_name = destination_object_name(str(row.destination_uri))
        remote = existing.get(object_name)
        rec = receipts.get(str(row.destination_uri))
        status = "TO_UPLOAD"
        if remote is not None:
            verified = False
            if rec is not None:
                verified = (
                    str(rec["manifest_sha256"]) == MANIFEST_SHA256
                    and str(rec["destination_uri"]) == str(row.destination_uri)
                    and int(rec["size_bytes"]) == int(remote["size"])
                    and str(rec["generation"]) == str(remote["generation"])
                    and str(rec["crc32c"]) == str(remote["crc32c"])
                )
            status = "ALREADY_UPLOADED_VERIFIED" if verified else "EXISTING_UNVERIFIED"
        rows.append({
            "manifest_ordinal": ordinal,
            "component": row.component,
            "source_path": row.source_path,
            "relative_path": row.relative_path,
            "destination_uri": row.destination_uri,
            "object_name": object_name,
            "size_bytes": int(row.size_bytes),
            "plan_status": status,
        })
    classified = pd.DataFrame(rows)
    existing_unverified = classified[classified["plan_status"] == "EXISTING_UNVERIFIED"]
    if not existing_unverified.empty:
        raise RuntimeError(f"Existen objetos planificados sin receipt verificable: {existing_unverified.head(10).to_dict('records')}")
    return classified

classified_manifest_df = classify_manifest_rows(manifest_df, existing_objects, receipt_df)
already_uploaded_verified = int((classified_manifest_df["plan_status"] == "ALREADY_UPLOADED_VERIFIED").sum())

candidate_df = classified_manifest_df[
    classified_manifest_df["component"].isin(TARGET_COMPONENTS)
    & (classified_manifest_df["plan_status"] != "ALREADY_UPLOADED_VERIFIED")
].sort_values("manifest_ordinal")
selected_df = candidate_df.head(MAX_FILES_THIS_RUN).copy()
selected_files = int(len(selected_df))
selected_bytes = int(selected_df["size_bytes"].sum()) if selected_files else 0

if already_uploaded_verified == 0:
    selected_counts = selected_df["component"].value_counts().to_dict()
    if selected_files != 2 or selected_bytes != EXPECTED_INITIAL_SELECTED_BYTES:
        raise RuntimeError(f"Seleccion inicial inesperada: files={selected_files} bytes={selected_bytes}")
    for component in TARGET_COMPONENTS:
        if int(selected_counts.get(component, 0)) != 1:
            raise RuntimeError(f"Seleccion inicial no contiene exactamente 1 archivo de {component}")

print(json.dumps({
    "selected_files": selected_files,
    "selected_bytes": selected_bytes,
    "already_uploaded_verified": already_uploaded_verified,
    "selection": selected_df[["component", "relative_path", "destination_uri", "size_bytes"]].to_dict("records"),
}, indent=2))

{
  "selected_files": 0,
  "selected_bytes": 0,
  "already_uploaded_verified": 2,
  "selection": []
}


## Confirmacion explicita

El token combina cantidad, bytes, bucket y SHA abreviado del manifest. Si el token no coincide, no se ejecuta ninguna escritura remota.

In [9]:
CONFIRMATION_TOKEN = f"UPLOAD_{selected_files}_FILES_{selected_bytes}_BYTES_TO_{BUCKET_NAME}_{MANIFEST_SHA256[:12]}"
UPLOAD_ARMED = bool(
    UPLOAD_ENABLED is True
    and CONFIRM_UPLOAD == CONFIRMATION_TOKEN
    and LOCAL_VALIDATIONS_READY
    and SOURCE_FILES_READY
    and BUCKET_READY
    and not unplanned_existing_objects
    and selected_files > 0
    and selected_files <= MAX_FILES_THIS_RUN
)

print(json.dumps({
    "required_confirmation_token": CONFIRMATION_TOKEN,
    "UPLOAD_ENABLED": UPLOAD_ENABLED,
    "UPLOAD_ARMED": UPLOAD_ARMED,
}, indent=2))

if not UPLOAD_ARMED:
    print("Carga no armada. Validaciones completadas; no se ejecutara ninguna escritura remota con la configuracion actual.")

{
  "required_confirmation_token": "UPLOAD_0_FILES_0_BYTES_TO_pfi-rm-lumbar-artifacts-2026-ef_fa54c89a278d",
  "UPLOAD_ENABLED": true,
  "UPLOAD_ARMED": false
}
Carga no armada. Validaciones completadas; no se ejecutara ninguna escritura remota con la configuracion actual.


## Carga controlada

La unica llamada de escritura remota permitida esta encapsulada en `upload_one_manifest_row`. Usa precondicion de generacion cero, checksum automatico y verificacion inmediata por metadata remota.

In [10]:
def append_failure(failures: pd.DataFrame, row: pd.Series, error_type: str, error_message: str) -> pd.DataFrame:
    item = {
        "manifest_sha256": MANIFEST_SHA256,
        "component": row["component"],
        "relative_path": row["relative_path"],
        "destination_uri": row["destination_uri"],
        "error_type": error_type,
        "error_message": str(error_message)[:500],
        "failed_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    failures = pd.concat([failures, pd.DataFrame([item])], ignore_index=True)
    save_failures(failures)
    return failures


def upload_one_manifest_row(bucket: Any, row: pd.Series) -> dict[str, Any]:
    source_path = Path(row["source_path"])
    if not source_path.exists() or source_path.is_symlink() or not source_path.is_file():
        raise ValueError(f"Fuente invalida: component={row['component']} relative_path={row['relative_path']}")
    if source_path.stat().st_size != int(row["size_bytes"]):
        raise ValueError(f"Size mismatch antes de cargar: component={row['component']} relative_path={row['relative_path']}")
    object_name = destination_object_name(str(row["destination_uri"]))
    blob = bucket.blob(object_name)
    blob.metadata = {
        "pfi_manifest_sha256": MANIFEST_SHA256,
        "pfi_component": str(row["component"]),
        "pfi_relative_path": str(row["relative_path"]),
        "pfi_source_size": str(int(row["size_bytes"])),
        "pfi_upload_notebook": "40",
    }
    blob.upload_from_filename(
        str(source_path),
        if_generation_match=0,
        checksum="auto",
        timeout=900,
    )
    blob.reload()
    if int(blob.size or 0) != int(row["size_bytes"]):
        raise ValueError(f"Verificacion de size fallo: {object_name}")
    if blob.generation is None or blob.crc32c is None:
        raise ValueError(f"Verificacion remota incompleta: {object_name}")
    return {
        "manifest_sha256": MANIFEST_SHA256,
        "component": row["component"],
        "source_path": row["source_path"],
        "relative_path": row["relative_path"],
        "destination_uri": row["destination_uri"],
        "object_name": object_name,
        "size_bytes": int(row["size_bytes"]),
        "crc32c": blob.crc32c,
        "md5_hash": blob.md5_hash,
        "generation": str(blob.generation),
        "updated": blob.updated.isoformat() if blob.updated is not None else "",
        "uploaded_at_utc": datetime.now(timezone.utc).isoformat(),
        "upload_status": "UPLOADED_VERIFIED",
    }

uploaded_receipts: list[dict[str, Any]] = []
if UPLOAD_ARMED:
    try:
        from google.api_core.exceptions import PreconditionFailed
        from google.cloud.storage.exceptions import DataCorruption
    except ImportError as exc:
        raise RuntimeError("Faltan excepciones de google-cloud-storage/google-api-core en el entorno.") from exc

    for row in selected_df.to_dict("records"):
        row_series = pd.Series(row)
        try:
            receipt_item = upload_one_manifest_row(bucket, row_series)
            uploaded_receipts.append(receipt_item)
            receipt_df = pd.concat([receipt_df, pd.DataFrame([receipt_item])], ignore_index=True)
            save_receipt(receipt_df)
            save_state({
                "manifest_sha256": MANIFEST_SHA256,
                "last_uploaded_destination_uri": row["destination_uri"],
                "updated_at_utc": datetime.now(timezone.utc).isoformat(),
            })
            print(f"Cargado y verificado: component={row['component']} relative_path={row['relative_path']}")
        except (PreconditionFailed, DataCorruption) as exc:
            failures_df = append_failure(failures_df, row_series, type(exc).__name__, str(exc))
            print(f"Fallo controlado: component={row['component']} relative_path={row['relative_path']} error={type(exc).__name__}")
            if FAIL_FAST:
                raise
        except Exception as exc:
            failures_df = append_failure(failures_df, row_series, type(exc).__name__, str(exc))
            print(f"Fallo: component={row['component']} relative_path={row['relative_path']} error={type(exc).__name__}")
            if FAIL_FAST:
                raise
else:
    print("UPLOAD_ARMED=False; se omite la celda de carga controlada.")

UPLOAD_ARMED=False; se omite la celda de carga controlada.


## Verificacion posterior

Luego de un lote armado, vuelve a listar `datasets/` una sola vez y contrasta los objetos cargados contra el receipt persistido.

In [11]:
def verify_uploaded_batch(client, receipt: pd.DataFrame, uploaded_items: list[dict[str, Any]]) -> int:
    if not uploaded_items:
        return 0
    current_objects = {
        blob.name: {
            "size": int(blob.size or 0),
            "crc32c": blob.crc32c,
            "generation": str(blob.generation) if blob.generation is not None else None,
        }
        for blob in client.list_blobs(BUCKET_NAME, prefix=GCS_PREFIX)
    }
    planned_objects = set(manifest_df["destination_uri"].map(destination_object_name))
    extras = []
    for object_name, remote in current_objects.items():
        if object_name in ALLOWED_ZERO_BYTE_PLACEHOLDERS and int(remote["size"]) == 0:
            continue
        if object_name not in planned_objects:
            extras.append(object_name)
    if extras:
        raise RuntimeError(f"Aparecieron objetos ajenos bajo {GCS_PREFIX}: {extras[:10]}")

    verified = 0
    for item in uploaded_items:
        remote = current_objects.get(item["object_name"])
        if remote is None:
            raise RuntimeError(f"Objeto recien cargado no encontrado: {item['object_name']}")
        if int(remote["size"]) != int(item["size_bytes"]):
            raise RuntimeError(f"Size final no coincide: {item['object_name']}")
        if str(remote["generation"]) != str(item["generation"]):
            raise RuntimeError(f"Generation final no coincide: {item['object_name']}")
        if str(remote["crc32c"]) != str(item["crc32c"]):
            raise RuntimeError(f"crc32c final no coincide: {item['object_name']}")
        verified += 1
    return verified

verified_this_run = verify_uploaded_batch(storage_client, receipt_df, uploaded_receipts)
print(json.dumps({"verified_this_run": verified_this_run}, indent=2))

{
  "verified_this_run": 0
}


## Resumen final

Resume el lote seleccionado, las cargas verificadas y el estado restante del manifest.

In [12]:
uploaded_this_run = len(uploaded_receipts)
failures_count = int(len(failures_df))
remaining_manifest_files = int(len(manifest_df) - len(receipt_df))
UPLOAD_BATCH_SUCCESS = bool(
    UPLOAD_ARMED
    and uploaded_this_run == selected_files
    and verified_this_run == selected_files
    and failures_count == 0
)

final_summary = {
    "manifest_sha256": MANIFEST_SHA256,
    "target_components": list(TARGET_COMPONENTS),
    "max_files_this_run": MAX_FILES_THIS_RUN,
    "selected_files": selected_files,
    "selected_bytes": selected_bytes,
    "uploaded_this_run": uploaded_this_run,
    "verified_this_run": verified_this_run,
    "already_uploaded_verified": already_uploaded_verified,
    "failures": failures_count,
    "remaining_manifest_files": remaining_manifest_files,
    "placeholder_count": len(allowed_placeholders),
    "UPLOAD_BATCH_SUCCESS": UPLOAD_BATCH_SUCCESS,
}
print(json.dumps(final_summary, indent=2))

{
  "manifest_sha256": "fa54c89a278d850021c0f91c0a27b3b5211c86301c9e4f125d75d517f39b793b",
  "target_components": [
    "metadata_e5",
    "metadata_e9"
  ],
  "max_files_this_run": 2,
  "selected_files": 0,
  "selected_bytes": 0,
  "uploaded_this_run": 0,
  "verified_this_run": 0,
  "already_uploaded_verified": 2,
  "failures": 0,
  "remaining_manifest_files": 2056,
  "placeholder_count": 1,
  "UPLOAD_BATCH_SUCCESS": false
}


## Proximas etapas

La primera ejecucion real debe limitarse a metadata E5/E9. SPIDER y AXIAL requieren tickets posteriores, con ajuste explicito de `TARGET_COMPONENTS` y limites de lote luego de revisar los receipts y la evidencia de esta etapa.